In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/ieee-fraud-detection/sample_submission.csv
/kaggle/input/ieee-fraud-detection/test_identity.csv
/kaggle/input/ieee-fraud-detection/train_identity.csv
/kaggle/input/ieee-fraud-detection/test_transaction.csv
/kaggle/input/ieee-fraud-detection/train_transaction.csv


In [2]:
!pip install -q scikit-learn==1.3.2 imbalanced-learn==0.11.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 76.3 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.6/235.6 kB 10.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
nilearn 0.11.1 requires scikit-learn>=1.4.0, but you have scikit-learn 1.3.2 which is incompatible.
bigframes 1.36.0 requires rich<14,>=12.4.4, but you have rich 14.0.0 which is incompatible.


In [3]:
!pip install -q category_encoders

In [4]:
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import OneHotEncoder
from category_encoders.woe import WOEEncoder

from imblearn.under_sampling import RandomUnderSampler

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler

from imblearn.pipeline import Pipeline  

import gc

from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score, roc_auc_score

In [5]:
train_identity = pd.read_csv("/kaggle/input/ieee-fraud-detection/train_identity.csv")
train_transaction = pd.read_csv("/kaggle/input/ieee-fraud-detection/train_transaction.csv")

test_identity = pd.read_csv("/kaggle/input/ieee-fraud-detection/test_identity.csv")
test_transaction = pd.read_csv("/kaggle/input/ieee-fraud-detection/test_transaction.csv")

# Cleaning & Feature Engineering

In [6]:
# Merge transaction and identity
train = train_transaction.merge(train_identity, how='left', on='TransactionID')
test = test_transaction.merge(test_identity, how='left', on='TransactionID')

# Drop TransactionID (only keep if needed for identification)
test_ID = test['TransactionID']
train.drop('TransactionID', axis=1, inplace=True)
test.drop('TransactionID', axis=1, inplace=True)

# Set target
y_train = train['isFraud']
X_train = train.drop('isFraud', axis=1)

In [7]:
X_train.shape, y_train.shape

((590540, 432), (590540,))

In [8]:
# Get columns that are present in both train and test
common_columns = X_train.columns.intersection(test.columns)

# Keep only those columns in both datasets
X_train = X_train[common_columns]
test = test[common_columns]

In [9]:
num_cols = X_train.columns[X_train.dtypes != 'object'].tolist()
len(num_cols)

378

In [10]:
cat_cols = X_train.columns[X_train.dtypes == 'object'].tolist()
len(cat_cols)

16

In [11]:
cat_unique = X_train[cat_cols].nunique()

In [12]:
for df in [X_train]:
    df['TransactionDT_days'] = df['TransactionDT'] / (3600 * 24)
    df['TransactionDT_hours'] = df['TransactionDT'] / 3600
    df['Transaction_hour'] = (df['TransactionDT'] // 3600) % 24
    df['Transaction_day'] = (df['TransactionDT'] // (3600 * 24)) % 7

In [13]:
missing = X_train.isnull().mean() * 100
missing = missing.sort_values(ascending=False) 

In [14]:
# 1. Drop columns with >90% missing
drop_cols = missing[missing > 90].index.tolist()
X_train.drop(columns=drop_cols, inplace=True)
# X_test.drop(columns=drop_cols, inplace=True)

# 2. Fill 50-90% missing columns
cols_50_90 = missing[(missing > 50) & (missing <= 90)].index.tolist()
for col in cols_50_90:
    if X_train[col].dtype == 'object':
        X_train.loc[:, col] = X_train[col].fillna('missing')
    else:
        X_train.loc[:, col] = X_train[col].fillna(-999)

# 3. Fill 10-50% missing columns
cols_10_50 = missing[(missing > 10) & (missing <= 50)].index.tolist()
for col in cols_10_50:
    if X_train[col].dtype == 'object':
        X_train.loc[:, col] = X_train[col].fillna('missing')
    else:
        X_train.loc[:, col] = X_train[col].fillna(-999)

# 4. Fill <10% missing columns
cols_under_10 = missing[(missing > 0) & (missing <= 10)].index.tolist()
for col in cols_under_10:
    if X_train[col].dtype == 'object':
        mode = X_train[col].mode()[0]
        X_train.loc[:, col] = X_train[col].fillna(mode)
    else:
        median = X_train[col].median()
        X_train.loc[:, col] = X_train[col].fillna(median)


In [15]:
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

X_train_split = X_train_split.reset_index(drop=True)
X_val_split = X_val_split.reset_index(drop=True)

y_train_split = y_train_split.reset_index(drop=True)
y_val_split = y_val_split.reset_index(drop=True)

In [16]:
woe_columns = list(cat_unique[cat_unique > 3].index)
one_hot_columns = list(cat_unique[cat_unique <= 3].index)

In [17]:
ohe = OneHotEncoder(sparse=False, drop='first', handle_unknown='ignore')
ohe.fit(X_train_split[one_hot_columns])

X_train_ohe = pd.DataFrame(
    ohe.transform(X_train_split[one_hot_columns]),
    columns=ohe.get_feature_names_out(one_hot_columns),
    index=X_train_split.index
)

X_val_ohe = pd.DataFrame(
    ohe.transform(X_val_split[one_hot_columns]),
    columns=ohe.get_feature_names_out(one_hot_columns),
    index=X_val_split.index
)

# X_test_ohe = pd.DataFrame(
#     ohe.transform(X_test[one_hot_columns]),
#     columns=ohe.get_feature_names_out(one_hot_columns),
#     index=X_test.index
# )

/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:975: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


In [18]:
woe = WOEEncoder()
woe.fit(X_train_split[woe_columns], y_train_split)

X_train_woe = pd.DataFrame(
    woe.transform(X_train_split[woe_columns]),
    columns=woe_columns,
    index=X_train_split.index
)

X_val_woe = pd.DataFrame(
    woe.transform(X_val_split[woe_columns]),
    columns=woe_columns,
    index=X_val_split.index
)

# X_test_woe = pd.DataFrame(
#     woe.transform(X_test[woe_columns]),
#     columns=woe_columns,
#     index=X_test.index
# )

In [19]:
# Drop encoded categorical columns from original data using .loc[] to avoid copying
X_train_split.drop(columns=woe_columns + one_hot_columns, inplace=True)
X_val_split.drop(columns=woe_columns + one_hot_columns, inplace=True)
# X_test.drop(columns=woe_columns + one_hot_columns, inplace=True)

# Combine everything using .loc[] for column selection
X_train_encoded = pd.concat([X_train_split, X_train_ohe, X_train_woe], axis=1)
X_val_encoded = pd.concat([X_val_split, X_val_ohe, X_val_woe], axis=1)
# X_test_encoded = pd.concat([X_test, X_test_ohe, X_test_woe], axis=1)

In [20]:
gc.collect()

0

# Feature Selection

In [21]:
# Step 1: Compute correlation matrix on training set
corr_matrix = X_train_encoded.corr().abs()

# Step 2: Create a mask for the upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

# Step 3: Find feature pairs with correlation greater than the threshold
threshold = 0.95
high_corr_pairs = [
    (corr_matrix.columns[i], corr_matrix.columns[j], corr_matrix.iloc[i, j])
    for i in range(len(corr_matrix.columns))
    for j in range(i+1, len(corr_matrix.columns))
    if corr_matrix.iloc[i, j] > threshold
]

# Step 4: Choose feature to drop (one with lower correlation with target)
features_to_drop = []

for feat1, feat2, _ in high_corr_pairs:
    corr1 = abs(X_train_encoded[feat1].corr(y_train))
    corr2 = abs(X_train_encoded[feat2].corr(y_train))
    
    # Drop feature with lower correlation to target
    if corr1 < corr2:
        features_to_drop.append(feat1)
    else:
        features_to_drop.append(feat2)

# Step 5: Remove duplicates
features_to_drop = list(set(features_to_drop))

In [22]:
len(features_to_drop)

276

In [23]:
len(X_train_encoded.columns)

407

In [24]:
# Final: Drop features from the training and test sets
X_train_filtered = X_train_encoded.drop(columns=features_to_drop)
X_val_filtered = X_val_encoded.drop(columns=features_to_drop)
#X_test_filtered = X_test_encoded.drop(columns=features_to_drop)

print(f"\nDropped {len(features_to_drop)} highly correlated features.")


Dropped 276 highly correlated features.


In [25]:
del corr_matrix

In [26]:
del train_transaction
del train_identity 
del test_transaction
del test_identity

In [27]:
del X_train_split
del X_val_split

del X_train
del y_train

# Training

In [42]:
# Define scalers and model
scalers = [
    StandardScaler(), 
    MinMaxScaler(), 
    None
]

model = {
    "Decision Tree": {
        "model": DecisionTreeClassifier(random_state=42, class_weight='balanced'),
        "params": {
            "scaler": scalers,
            "model__max_depth": [5, 7, 10],
            "model__min_samples_split": [2, 5],
            "model__min_samples_leaf": [1, 2],
            "model__criterion": ['gini', 'entropy']  
        }
    }
}

# Cross-validation
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Loop through models
for name, config in model.items():
    print(f"Training {name}...")
    
    # Define pipeline with placeholders
    pipeline = Pipeline([
        ("scaler", StandardScaler()),  
       # ("under", RandomUnderSampler(random_state=42)),
        ("model", config["model"])
    ])

    # Replace `scaler` param name properly in param_grid
    param_grid = config["params"]
    param_grid["scaler"] = scalers  # in pipeline, first step is named "scaler"

    # Grid search
    grid_search = GridSearchCV(
        pipeline,
        param_grid=config["params"],
        scoring='f1',  # Use 'f1', 'recall', or 'roc_auc' for imbalanced data
        cv=kfold,
        n_jobs=-1,
        return_train_score=True,
        verbose=1
    )

    grid_search.fit(X_train_filtered, y_train_split)

    print(f"Best parameters for {name}: {grid_search.best_params_}")
    print(f"Best cross-validation f1: {grid_search.best_score_:.4f}\n")


Training Decision Tree...
Fitting 5 folds for each of 72 candidates, totalling 360 fits
Best parameters for Decision Tree: {'model__criterion': 'gini', 'model__max_depth': 10, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'scaler': MinMaxScaler()}
Best cross-validation f1: 0.2705



In [43]:
pipeline = Pipeline([
    ('scaler', MinMaxScaler()),
    #('undersampler', RandomUnderSampler(random_state=42)),
    ('classifier', DecisionTreeClassifier(
        class_weight='balanced',
        random_state=42,
        criterion='gini',
        max_depth=10,
        min_samples_leaf=1,
        min_samples_split=2
    ))
])

# Fit the pipeline on training data
pipeline.fit(X_train_filtered, y_train_split)

# Predict on train and validation
y_train_pred = pipeline.predict(X_train_filtered)
y_val_pred = pipeline.predict(X_val_filtered)
y_train_proba = pipeline.predict_proba(X_train_filtered)[:, 1]
y_val_proba = pipeline.predict_proba(X_val_filtered)[:, 1]

In [44]:
# Train metrics
train_f1 = f1_score(y_train_split, y_train_pred)
train_recall = recall_score(y_train_split, y_train_pred)
train_precision = precision_score(y_train_split, y_train_pred)
train_accuracy = accuracy_score(y_train_split, y_train_pred)
train_auc = roc_auc_score(y_train_split, y_train_proba)

# Validation metrics
val_f1 = f1_score(y_val_split, y_val_pred)
val_recall = recall_score(y_val_split, y_val_pred)
val_precision = precision_score(y_val_split, y_val_pred)
val_accuracy = accuracy_score(y_val_split, y_val_pred)
val_auc = roc_auc_score(y_val_split, y_val_proba)

In [45]:
print(f"Train F1 Score: {train_f1:.4f}")
print(f"Train Recall: {train_recall:.4f}")
print(f"Train Precision: {train_precision:.4f}")
print(f"Train Accuracy: {train_accuracy:.4f}")
print(f"Train AUC-ROC: {train_auc:.4f}")

Train F1 Score: 0.2968
Train Recall: 0.7679
Train Precision: 0.1840
Train Accuracy: 0.8727
Train AUC-ROC: 0.8947


In [46]:
print(f"Validation F1 Score: {val_f1:.4f}")
print(f"Validation Recall: {val_recall:.4f}")
print(f"Validation Precision: {val_precision:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"Validation AUC-ROC: {val_auc:.4f}")

Validation F1 Score: 0.2833
Validation Recall: 0.7309
Validation Precision: 0.1757
Validation Accuracy: 0.8706
Validation AUC-ROC: 0.8649


# MLflow Logging

In [38]:
!pip install -q dagshub mlflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.1/260.1 kB 6.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 32.6 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 74.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 692.3/692.3 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.2/203.2 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 3.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed

In [47]:
import dagshub
import mlflow

In [48]:
dagshub.init(repo_owner='mrekh21', repo_name='Fraud_Detection', mlflow=True)

Initialized MLflow to track repo "mrekh21/Fraud_Detection"

Repository mrekh21/Fraud_Detection initialized!

In [49]:
experiment_name = "DecisionTree_Training"
run_name = "cross_validation_6"

# Set the experiment (creates if it doesn't exist)
mlflow.set_experiment(experiment_name)

# Start an MLflow run with a specific name
with mlflow.start_run(run_name=run_name):

    # Log hyperparameters
    mlflow.log_params({
        'model': "DecisionTreeClassifier",
        'scaler': "MinMaxScaler",
    #    'sampling': 'RandomUnderSampler',
        'class_weight': 'balanced',
        'random_state': 42,
        'criterion': 'gini',
        'max_depth': 10,
        'min_samples_leaf': 1,
        'min_samples_split': 2
    })

    # Log performance metric
    mlflow.log_metric("train_f1", train_f1)
    mlflow.log_metric("train_recall", train_recall)
    mlflow.log_metric("train_precision", train_precision)
    mlflow.log_metric("train_accuracy", train_accuracy)
    mlflow.log_metric("train_auc_roc", train_auc)
    
    mlflow.log_metric("val_f1", val_f1)
    mlflow.log_metric("val_recall", val_recall)
    mlflow.log_metric("val_precision", val_precision)
    mlflow.log_metric("val_accuracy", val_accuracy)
    mlflow.log_metric("val_auc_roc", val_auc)


#    mlflow.set_tag("Description", "Training Decision Tree with undersampling and MinMaxScaler. train_transaction + train_indentity, OneHot + WOE, missing values handled with 'missing'/mode or -999/median")
    mlflow.set_tag("Description", "Training Decision Tree with StandardScaler and class weight. train_transaction + train_indentity, OneHot + WOE, missing values handled with 'missing'/mode or -999/median, correlation filter(threshold=0.95)")

print("Pipeline logged successfully!")

🏃 View run cross_validation_6 at: https://dagshub.com/mrekh21/Fraud_Detection.mlflow/#/experiments/1/runs/7a184627eae54132ad44444f835e213a
🧪 View experiment at: https://dagshub.com/mrekh21/Fraud_Detection.mlflow/#/experiments/1
Pipeline logged successfully!
